In [4]:
import kagglehub
import pandas as pd
import os
from datetime import datetime


path = kagglehub.dataset_download("antonkozyriev/game-recommendations-on-steam")
print("Path to dataset files:", path)


games_raw = pd.read_csv(os.path.join(path, "games.csv"))
users_raw = pd.read_csv(os.path.join(path, "users.csv"))
recommendations_raw = pd.read_csv(os.path.join(path, "recommendations.csv"))


games_df = games_raw.copy()

interactions_df = recommendations_raw.merge(users_raw, on="user_id", how="left")


Path to dataset files: /Users/aregayvazyan/.cache/kagglehub/datasets/antonkozyriev/game-recommendations-on-steam/versions/28


In [8]:
games_clean = games_df.copy()

games_clean = games_clean.drop(
    columns=["win", "mac", "linux", "steam_deck", "discount", "price_original", "title"],
    errors="ignore"
)

def parse_date(date_str):
    try:
        return datetime.strptime(date_str, "%Y-%m-%d")
    except (ValueError, TypeError):
        return None

games_clean["date_release"] = games_clean["date_release"].apply(parse_date)

print("Rating categories:", games_clean["rating"].unique())

rating_map = {
    "Overwhelmingly Negative": 0,
    "Very Negative": 1,
    "Negative": 2,
    "Mostly Negative": 3,
    "Mixed": 4,
    "Mostly Positive": 5,
    "Positive": 6,
    "Very Positive": 7,
    "Overwhelmingly Positive": 8
}

games_clean["rating"] = games_clean["rating"].map(rating_map)


before = games_clean.shape[0]
games_clean = games_clean[games_clean["user_reviews"] >= 10]
after = games_clean.shape[0]
print(f"Dropped {before - after} games with <10 reviews ({before} -> {after})")

Rating categories: ['Very Positive' 'Positive' 'Mixed' 'Mostly Positive'
 'Overwhelmingly Positive' 'Negative' 'Mostly Negative'
 'Overwhelmingly Negative' 'Very Negative']
Dropped 0 games with <10 reviews (50872 -> 50872)


In [9]:
print(games_clean.shape)
games_clean.head()

(50872, 6)


,app_id,date_release,rating,positive_ratio,user_reviews,price_final
0,13500,2008-11-21,7,84,2199,9.99
1,22364,2011-08-03,6,85,21,2.99
2,113020,2013-04-24,7,92,3722,14.99
3,226560,2014-11-18,4,61,873,14.99
4,249050,2014-10-27,7,88,8784,11.99


In [ ]:
from scipy.sparse import csr_matrix

user_ids = interactions_df["user_id"].unique()
app_ids = interactions_df["app_id"].unique()

user_mapper = {uid: i for i, uid in enumerate(user_ids)}
game_mapper = {aid: i for i, aid in enumerate(app_ids)}

user_inv_mapper = {i: uid for uid, i in user_mapper.items()}
game_inv_mapper = {i: aid for aid, i in game_mapper.items()}

interactions_df["user_idx"] = interactions_df["user_id"].map(user_mapper)
interactions_df["app_idx"] = interactions_df["app_id"].map(game_mapper)


X = csr_matrix(
    (interactions_df["is_recommended"].astype(int),
     (interactions_df["app_idx"], interactions_df["user_idx"])),
    shape=(len(game_mapper), len(user_mapper))
)


In [22]:
sparsity = X.count_nonzero()/(X.shape[0]*X.shape[1])

print(f"Matrix sparsity: {round(sparsity*100,2)}%")

Matrix sparsity: 0.01%


In [23]:
from scipy.sparse import save_npz

save_npz('data/user_item_matrix.npz', X)

In [ ]:
from sklearn.neighbors import NearestNeighbors

def find_similar_games(app_id, X_sparse, k, metric='cosine', show_distance=False):
    neighbour_ids = []
    neighbour_dists = []

    game_ind = game_mapper[app_id]
    game_vec = X_sparse[game_ind]

    if isinstance(game_vec, np.ndarray):
        game_vec = game_vec.reshape(1, -1)

    kNN = NearestNeighbors(n_neighbors=k + 1, algorithm="brute", metric=metric)
    kNN.fit(X_sparse)

    distances, indices = kNN.kneighbors(game_vec, return_distance=True)

    for i in range(1, k + 1): 
        n = indices[0, i]
        neighbour_ids.append(game_inv_mapper[n])
        neighbour_dists.append(distances[0, i])

    if show_distance:
        return list(zip(neighbour_ids, neighbour_dists))
    return neighbour_ids


In [29]:
import numpy as np

games_lookup = games_df[["app_id", "title"]].copy()
sample_app_id = app_ids[0]
similar = find_similar_games(sample_app_id, X, k=10)

print(f"Games similar to app_id {sample_app_id}:")
print(games_lookup[games_lookup["app_id"] == sample_app_id])

print("\nSimilar games:")
print(games_lookup[games_lookup["app_id"].isin(similar)])

Games similar to app_id 975370:
       app_id           title
48244  975370  Dwarf Fortress

Similar games:
        app_id               title
6432    688060           Odd Realm
12740   294100            RimWorld
12802  1158310  Crusader Kings III
14290   233860              Kenshi
16612   351700        UnReal World
47674   333640        Caves of Qud
47793   427520            Factorio
48165   881100               Noita
48401  1154840       Shadow Empire
48412  1162750        Songs of Syx
